# SymptoGraph-LLM: download the datasets straight into Google Drive

Run this in Colab. It downloads all eight open datasets on Colab's own network
(fast), then writes them to your Drive. Nothing is downloaded or uploaded from
your own machine.

**How it works.** The data is first staged on the Colab VM's local disk, which is
much faster than writing through the Drive mount. The Reddit Mental Health
Dataset is then slimmed from 3.14 GB to 0.41 GB by dropping 346
precomputed tf-idf and readability columns that this project does not use, and
only the result is copied to Drive.

| | Downloaded to the VM | Ends up in Drive |
|---|---|---|
| Default (`KEEP_RAW_RMHD = False`) | ~3.1 GB | **~0.6 GB** |
| Keeping the raw CSVs (`True`) | ~3.1 GB | ~3.7 GB |

Expect roughly 10–20 minutes, mostly the Drive copy. A free Drive account has
15 GB, so the default comfortably fits.

If the runtime disconnects, just run the cells again. Downloads resume: finished
files are skipped and every Reddit Mental Health file is checked against its
Zenodo checksum.

## 1. Mount Drive and set the paths

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Where the data should end up (inside your Drive)
DRIVE_DIR = '/content/drive/MyDrive/SymptoGraph-LLM/data'
# Fast local scratch space on the Colab VM
STAGE_DIR = '/content/stage'
# Set to True to also keep the full 3 GB raw Reddit CSVs in Drive
KEEP_RAW_RMHD = False

os.environ.update(DRIVE_DIR=DRIVE_DIR, STAGE_DIR=STAGE_DIR,
                  DATA_DIR=STAGE_DIR, JOBS='8')
os.makedirs(STAGE_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
print('staging in', STAGE_DIR, '-> Drive folder', DRIVE_DIR)

## 2. Get the download scripts

The repo itself is tiny, so it is cloned to the VM rather than to Drive.

In [ ]:
REPO = 'https://github.com/sid-2k6/SymptoGraph-LLM.git'
if os.path.isdir('/content/SymptoGraph-LLM/.git'):
    !git -C /content/SymptoGraph-LLM pull -q
else:
    !git clone -q $REPO /content/SymptoGraph-LLM
!ls /content/SymptoGraph-LLM/scripts

## 3. Download every dataset to the VM

Eight datasets: DepressionEmo, Reddit Mental Health, DepSeverity,
CSSRS-Reddit-500, IMHI, Dreaddit, IRF and LoST.

Useful variants, if you want to cut this down:

```python
os.environ['RMHD_FILTER'] = 'depression|anxiety|bipolar|suicidewatch'  # only these subreddits
os.environ['SKIP'] = 'imhi'                                           # skip a dataset entirely
```

In [ ]:
!bash /content/SymptoGraph-LLM/scripts/download_data.sh

## 4. Slim the Reddit Mental Health Dataset

Keeps only `subreddit`, `author`, `date` and `post`, gzipped. This is the step
that turns 3.14 GB into 0.41 GB (108 files, 1,107,302 posts). Skip this cell if you set
`KEEP_RAW_RMHD = True` and want the raw features.

In [ ]:
!python /content/SymptoGraph-LLM/scripts/slim_rmhd.py \
    --src "$STAGE_DIR/reddit_mental_health" \
    --dst "$STAGE_DIR/reddit_mental_health_slim"

## 5. Copy to Drive

`rsync` can be re-run safely: it only copies what is missing or changed.

In [ ]:
exclude = '' if KEEP_RAW_RMHD else "--exclude=reddit_mental_health/"
!rsync -ah --info=progress2 $exclude "$STAGE_DIR/" "$DRIVE_DIR/"
print('\nIn Drive now:')
!du -sh "$DRIVE_DIR"/* | sort -h
!du -sh "$DRIVE_DIR"

## 6. Check the data loads

Reads one file from each dataset out of Drive and prints its shape, so you know
the copy is intact before you rely on it.

In [ ]:
import csv, glob, gzip, json, os, sys
import pandas as pd
csv.field_size_limit(sys.maxsize)
D = DRIVE_DIR

def line_count(path):
    with open(path) as fh:
        return sum(1 for l in fh if l.strip())

checks = []
for split in ['train', 'val', 'test']:
    p = f'{D}/depressionemo/Dataset/{split}.json'
    checks.append((f'DepressionEmo {split}', line_count(p)))

checks.append(('DepSeverity', len(pd.read_csv(f'{D}/depseverity/Reddit_depression_dataset.csv'))))
checks.append(('CSSRS-500', len(pd.read_csv(
    f'{D}/cssrs_reddit_500/anonymized_dataset/500_Reddit_users_posts_labels.csv'))))
checks.append(('IRF', sum(len(pd.read_csv(f'{D}/irf/{s}_data.csv'))
                          for s in ['train', 'val', 'test'])))
checks.append(('LoST', len(pd.read_csv(f'{D}/lost/LoSTdataset.csv'))))
checks.append(('Dreaddit', sum(len(pd.read_parquet(f))
                               for f in glob.glob(f'{D}/dreaddit/*.parquet')))) 
slim = sorted(glob.glob(f'{D}/reddit_mental_health_slim/*.csv.gz'))
checks.append((f'Reddit MH slim ({len(slim)} files)',
               len(pd.read_csv(slim[0], compression='gzip')) if slim else 0))

for name, n in checks:
    print(f'{name:28s} {n:>8,} rows')

print('\nIMHI:', len(glob.glob(f'{D}/imhi/train_data/**/*.*', recursive=True)), 'train files,',
      len(glob.glob(f'{D}/imhi/test_data/**/*.*', recursive=True)), 'test files')

if slim:
    df = pd.read_csv(slim[0], compression='gzip')
    print('\nSample from', os.path.basename(slim[0]), '->', list(df.columns))
    print(df[['subreddit', 'date']].head(3).to_string(index=False))

## Using the data in later sessions

Mount Drive and read straight from `DRIVE_DIR`. Reading through the Drive mount
is slow, so for training copy what you need onto the VM's local disk first:

```python
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/data
!rsync -ah /content/drive/MyDrive/SymptoGraph-LLM/data/ /content/data/
```

Reminders on terms of use:

- Keep the data in a **private** Drive folder and do not share it publicly.
  DepressionEmo is "All Rights Reserved" and is licensed for research only.
- Paraphrase any example posts you quote in the paper, and do not attempt to
  identify users.
- `data/README.md` in the repo lists every dataset's source, licence and role.